# Performance Estimation: the subgradient method's worst case as an SDP

This notebook turns the **adversarial perspective** from the slides into a concrete,
solvable problem. We compute the *exact* worst-case accuracy of the **averaged
subgradient method** over nonsmooth convex $L$-Lipschitz functions, and check it against
the Lecture-8 bound
$$ f(x_A) - f(x_\star) \;\le\; E(h) \;=\; \frac{R^2 + L^2\sum_{i=0}^N h_i^2}{2\sum_{i=0}^N h_i}. $$

Throughout we fix $R = L = 1$ and $N = 10$.

## Setup (as on the slides)

Subgradient descent queries $g_i \in \partial f(x_i)$ and updates
$$ x_{i+1} = x_i - h_i g_i, \qquad x_A = \frac{\sum_{i=0}^N h_i\,x_i}{\sum_{i=0}^N h_i}. $$
The proof sees only the sampled data $f_i = f(x_i)$ and $g_i$, for $i \in \{0,\dots,N+1,A,\star\}$.
The optimal constant stepsize is $h_i = R/(L\sqrt{N+1})$, for which $E(h) = RL/\sqrt{N+1}$.

In [2]:
import numpy as np
import cvxpy as cp

R, L, N = 1.0, 1.0, 10
h   = np.full(N + 1, R / (L * np.sqrt(N + 1)))   # constant optimal stepsize h_0..h_N
lam = h / h.sum()                                # averaging weights lambda_i = h_i / sum_j h_j

E_h = (R**2 + L**2 * np.sum(h**2)) / (2 * np.sum(h))
print(f"E(h)         = {E_h:.6f}")
print(f"RL/sqrt(N+1) = {R * L / np.sqrt(N + 1):.6f}")

E(h)         = 0.301511
RL/sqrt(N+1) = 0.301511


## The Gram matrix $G$ and values $F$

Stack the vectors of interest as the columns of $P$ and let $G = P^\top P \succeq 0$ collect
their pairwise inner products,
$$ P = \big[\,x_0, \dots, x_N, x_{N+1},\, x_A,\, x_\star,\ g_0, \dots, g_{N+1},\, g_A\,\big],
   \qquad G = P^\top P . $$

We keep $x_A$ as its own column. The averaging identity $x_A = \sum_i \lambda_i x_i$ is then imposed
on $G$ as the **linear** equalities
$$ \big\langle x_A - \textstyle\sum_i \lambda_i x_i,\; p_k\big\rangle = 0 \ \text{ for every column } p_k \text{ of P},
   \qquad\text{i.e.}\qquad G\,\big(e_{x_A} - \textstyle\sum_i \lambda_i e_{x_i}\big) = 0 . $$

Every vector is a coordinate vector in this basis, and any inner product
$\langle a, b\rangle = a^\top G\,b$ is **linear in $G$**.

In [3]:
# --- column layout of P = [x_0..x_{N+1}, x_A, x_star, g_0..g_{N+1}, g_A] ---
idx, col = {}, 0
for i in range(N + 2):            # x_0, ..., x_N, x_{N+1}
    idx[("x", i)] = col; col += 1
idx[("x", "A")] = col; col += 1   # x_A
idx[("x", "*")] = col; col += 1   # x_star
for i in range(N + 2):            # g_0, ..., g_{N+1}
    idx[("g", i)] = col; col += 1
idx[("g", "A")] = col; col += 1   # g_A
m = col                           # number of columns of P

def vec(key):
    "coordinate vector e_key in the P-basis"
    v = np.zeros(m); v[idx[key]] = 1.0
    return v

x  = [vec(("x", i)) for i in range(N + 2)]    # x[0], ..., x[N+1]
xA = vec(("x", "A"))                           # x_A
xs = vec(("x", "*"))                           # x_star
g  = [vec(("g", i)) for i in range(N + 2)]     # g[0], ..., g[N+1]
gA = vec(("g", "A"))                           # g_A

# --- lifted decision variables ---
G = cp.Variable((m, m), symmetric=True)        # Gram matrix, G = P^T P
F = cp.Variable(N + 4)                          # F = (f_0, ..., f_{N+1}, f_A, f_star)
f, fA, fs = [F[i] for i in range(N + 2)], F[N + 2], F[N + 3]

def ip(a, b):
    "inner product <a, b> = a^T G b   (linear in G)"
    return a @ G @ b

## Constraints (exactly the adversarial-perspective slide)

Colour-coded as on the slide:

* <span style="color:#1f77b4">**blue**</span> &mdash; the descent recursion
  $\lVert x_{i+1}-x_\star\rVert^2 \le \lVert x_i-x_\star\rVert^2 - 2h_i\langle g_i, x_i-x_\star\rangle + h_i^2\lVert g_i\rVert^2$,
  together with the averaging identity $x_A = \sum_i \lambda_i x_i$;
* <span style="color:#2ca02c">**green**</span> &mdash; the start $\lVert x_0-x_\star\rVert^2 \le R^2$;
* <span style="color:#d62728">**red**</span> &mdash; the two families of subgradient inequalities and the
  Lipschitz bound $\lVert g_i\rVert^2 \le L^2$;
* and $G \succeq 0$, the only constraint tying $G$ back to genuine vectors.

In [4]:
cons = [G >> 0]

# blue: averaging x_A = sum_i lambda_i x_i, as linear equalities G (e_{x_A} - sum_i lambda_i e_{x_i}) = 0
avg = xA - sum(lam[i] * x[i] for i in range(N + 1))
cons += [G @ avg == np.zeros(m)]

# blue: descent recursion, i = 0..N  (the i = N step uses x_{N+1})
for i in range(N + 1):
    di, di1 = x[i] - xs, x[i + 1] - xs
    cons += [ip(di1, di1) <= ip(di, di) - 2 * h[i] * ip(g[i], di) + h[i]**2 * ip(g[i], g[i])]

# green: starting condition
cons += [ip(x[0] - xs, x[0] - xs) <= R**2]

# red: subgradient inequality between x_i and x_star, i = 0..N+1
for i in range(N + 2):
    cons += [ip(g[i], xs - x[i]) <= fs - f[i]]

# red: subgradient inequality between x_A and x_i   (sums to f_A <= sum_i lambda_i f_i)
for i in range(N + 1):
    cons += [ip(gA, x[i] - xA) <= f[i] - fA]

# red: Lipschitz bound, i = 0..N+1
for i in range(N + 2):
    cons += [ip(g[i], g[i]) <= L**2]

print(f"{len(cons)} constraints; G is {m} x {m}")

49 constraints; G is 27 x 27


In [5]:
prob = cp.Problem(cp.Maximize(fA - fs), cons)
pep_value = prob.solve(solver=cp.CLARABEL)

print(f"status         = {prob.status}")
print(f"PEP worst case = {pep_value:.6f}")
print(f"E(h)           = {E_h:.6f}")
print(f"difference     = {abs(pep_value - E_h):.2e}")

status         = optimal
PEP worst case = 0.301511
E(h)           = 0.301511
difference     = 4.50e-09


The analysis is tight. Given the information we used (the constraints), the worst-case objective can not be improved!

## Can we do better? Add *every* valid inequality

A convex $L$-Lipschitz function obeys the subgradient inequality between **every** ordered pair of
sampled points, $\langle g_i, x_j - x_i\rangle \le f_j - f_i$, and $\lVert g_i\rVert \le L$ at each.
The analysis above used only a handful of these. Let us throw in *all* of them and re-solve: if the
textbook bound were loose, more inequalities could only tighten the worst case.

In [6]:
# fresh copy of the variables so the dual multipliers are clean
G2 = cp.Variable((m, m), symmetric=True)
F2 = cp.Variable(N + 4)
f2, fA2, fs2 = [F2[i] for i in range(N + 2)], F2[N + 2], F2[N + 3]
ip2 = lambda a, b: a @ G2 @ b

pts  = list(range(N + 2)) + ["A", "*"]      # all sampled points  (have x_i, f_i)
gpts = list(range(N + 2)) + ["A"]           # points with a subgradient g_i
xof  = lambda i: x[i] if isinstance(i, int) else (xA if i == "A" else xs)
fof  = lambda i: f2[i] if isinstance(i, int) else (fA2 if i == "A" else fs2)
gof  = lambda i: g[i] if isinstance(i, int) else gA

ineq = []   # labelled scalar inequalities (the prunable ones)
# algorithm + problem structure
for i in range(N + 1):
    di, di1 = x[i] - xs, x[i + 1] - xs
    ineq.append((f"recursion step {i}",
        ip2(di1, di1) <= ip2(di, di) - 2*h[i]*ip2(g[i], di) + h[i]**2*ip2(g[i], g[i])))
ineq.append(("start  ||x0-x*||^2 <= R^2", ip2(x[0]-xs, x[0]-xs) <= R**2))
# EVERY pairwise subgradient inequality  <g_i, x_j - x_i> <= f_j - f_i
for i in gpts:
    for j in pts:
        if i == j: continue
        ineq.append((f"subgrad  g_{i}.(x_{j}-x_{i})",
                     ip2(gof(i), xof(j) - xof(i)) <= fof(j) - fof(i)))
# Lipschitz at every subgradient
for i in gpts:
    ineq.append((f"Lipschitz ||g_{i}||^2 <= L^2", ip2(gof(i), gof(i)) <= L**2))

# structural equality / PSD (not pruned)
avg_con = G2 @ avg == np.zeros(m)
cons2 = [G2 >> 0, avg_con] + [c for _, c in ineq]
prob2 = cp.Problem(cp.Maximize(fA2 - fs2), cons2)
val2 = prob2.solve(solver=cp.CLARABEL)

print(f"inequalities thrown in = {len(ineq)}")
print(f"worst case  = {val2:.6f}")
print(f"E(h)        = {E_h:.6f}   ->  no improvement, every added inequality is redundant")

inequalities thrown in = 194
worst case  = 0.301511
E(h)        = 0.301511   ->  no improvement, every added inequality is redundant


/tmp/ipykernel_438003/3205600029.py:34: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  val2 = prob2.solve(solver=cp.CLARABEL)


## Which inequalities does the proof actually use?

Nothing improved — but the SDP *dual* tells us **why**. By LP/SDP duality, a feasible set of
**dual multipliers** is exactly a proof: a nonnegative combination of the inequalities that adds up
to $f_A - f_\star \le E(h)$. Complementary slackness forces a zero multiplier on every inequality
that is slack at the worst case, so

> the inequalities with **nonzero multiplier** are the proof; all the **zero-multiplier** ones are
> redundant and can be dropped (for this fixed $N$).

This is the seed of the PEP framework: a *finite* list of inequalities suffices, and the dual picks
out the few that matter.

In [7]:
from collections import defaultdict

tol  = 1e-4
mult = [(lbl, float(np.ravel(c.dual_value)[0])) for lbl, c in ineq]
used = sorted([(lbl, d) for lbl, d in mult if abs(d) > tol], key=lambda t: -abs(t[1]))

print(f"{len(used)} of {len(ineq)} inequalities carry a nonzero multiplier -- the rest are redundant.\n")

cat = defaultdict(lambda: [0, 0])           # used / total, per family
for lbl, d in mult:
    k = lbl.split()[0]
    cat[k][0] += abs(d) > tol; cat[k][1] += 1
for k, (nz, total) in cat.items():
    print(f"  {k:10s}: {nz:3d} used / {total:3d}")

print("\nthe proof (inequalities with nonzero multiplier, sorted):")
for lbl, d in used:
    print(f"  mult = {d:+.4f}   {lbl}")

47 of 194 inequalities carry a nonzero multiplier -- the rest are redundant.

  recursion :  11 used /  11
  start     :   1 used /   1
  subgrad   :  23 used / 169
  Lipschitz :  12 used /  13

the proof (inequalities with nonzero multiplier, sorted):
  mult = +0.1508   recursion step 4
  mult = +0.1508   recursion step 3
  mult = +0.1508   recursion step 2
  mult = +0.1508   recursion step 5
  mult = +0.1508   recursion step 1
  mult = +0.1508   recursion step 6
  mult = +0.1508   recursion step 0
  mult = +0.1508   start  ||x0-x*||^2 <= R^2
  mult = +0.1508   recursion step 7
  mult = +0.1508   recursion step 8
  mult = +0.1508   recursion step 9
  mult = +0.0909   subgrad  g_A.(x_0-x_A)
  mult = +0.0909   subgrad  g_A.(x_1-x_A)
  mult = +0.0909   subgrad  g_A.(x_2-x_A)
  mult = +0.0909   subgrad  g_A.(x_3-x_A)
  mult = +0.0909   subgrad  g_A.(x_4-x_A)
  mult = +0.0909   subgrad  g_A.(x_5-x_A)
  mult = +0.0909   subgrad  g_A.(x_6-x_A)
  mult = +0.0909   subgrad  g_A.(x_7-x_A)
  mult